# ۴. قدرت کلی تیم‌ها و Elo زمانی

در این نوت‌بوک دو مفهوم جدا نگه داشته می‌شوند: قدرت کلی برگرفته از تاریخ کامل و Elo پویا که فقط از مسابقات قبلی فایل تاریخ‌دار استفاده می‌کند.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "DataSet2").is_dir())
ANALYSIS_ROOT = ROOT / "UCL_Analysis2"
OUTPUT = ANALYSIS_ROOT / "Output"
sys.path.insert(0, str(ANALYSIS_ROOT / "src"))
pd.set_option("display.max_columns", 100)
plt.style.use("seaborn-v0_8-whitegrid")
print("Repository root:", ROOT)


In [ ]:
alltime = pd.read_csv(OUTPUT / "alltime_team_strength.csv")
dynamic = pd.read_csv(OUTPUT / "dynamic_elo_final.csv")
enriched = pd.read_csv(OUTPUT / "detailed_matches_enriched.csv")

comparison = alltime[["Rank", "Team", "Matches", "win_rate", "points_per_match", "alltime_elo_proxy"]].merge(dynamic, on="Team", how="inner")
comparison["alltime_proxy_rank"] = comparison.alltime_elo_proxy.rank(ascending=False, method="min").astype(int)
comparison["dynamic_rank"] = comparison.dynamic_elo_final.rank(ascending=False, method="min").astype(int)
comparison["rank_change_dynamic_minus_proxy"] = comparison.alltime_proxy_rank - comparison.dynamic_rank
comparison.sort_values("dynamic_rank").head(20)


In [ ]:
print("Spearman correlation between all-time proxy and final dynamic Elo:",
      comparison[["alltime_elo_proxy", "dynamic_elo_final"]].corr(method="spearman").iloc[0,1].round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
top_alltime = alltime.nlargest(15, "alltime_elo_proxy").sort_values("alltime_elo_proxy")
axes[0].barh(top_alltime.Team, top_alltime.alltime_elo_proxy, color="#457b9d")
axes[0].set_title("All-time Elo-like strength (top 15)")
axes[0].set_xlabel("Elo-like rating")

top_dynamic = dynamic.head(15).sort_values("dynamic_elo_final")
axes[1].barh(top_dynamic.Team, top_dynamic.dynamic_elo_final, color="#2a9d8f")
axes[1].set_title("Final leakage-safe dynamic Elo (top 15)")
axes[1].set_xlabel("Dynamic Elo after last dated match")
fig.tight_layout()
fig.savefig(OUTPUT / "elo_rankings.png", dpi=180, bbox_inches="tight")
plt.show()


## مثال تفسیر

اگر Elo پیش از بازی میزبان ۱۶۵۰ و مهمان ۱۵۰۰ باشد، اختلاف `+150` است. انتظار Elo بدون وارد کردن امتیاز میزبانی برابر با `1 / (1 + 10^(-150/400)) ≈ 0.70` می‌شود. این مقدار نتیجه مورد انتظار (برد=۱، مساوی=۰٫۵، باخت=۰) است، نه مستقیماً احتمال برد.

In [ ]:
example_diff = 150
expected_score = 1 / (1 + 10 ** (-example_diff / 400))
print(f"Expected home result score for +150 Elo: {expected_score:.3f}")
display(enriched[["date", "home_team", "away_team", "home_elo_pre", "away_elo_pre", "elo_diff_pre", "elo_expected_home_score", "result"]].tail(10))


## محدودیت علمی

`alltime_elo_proxy` از کل تاریخ جدول استفاده می‌کند و برای توصیف قدرت کلی مناسب است، ولی اگر آن را برای پیش‌بینی یک بازی قدیمی استفاده کنیم اطلاعات آینده وارد مدل می‌شود. در مقابل، `home_elo_pre` و `away_elo_pre` فقط از بازی‌های قبل ساخته شده‌اند و leakage-safe هستند.